# California Residential Home Price Prediction
# Random Forest Model

This notebook trains a Random Forest Regressor and compares its performance against the Linear Regression baseline from the previous notebook.

**Evaluation metrics:** R2, MAPE, MdAPE

## Section 1 - Setup & Load Data

In [21]:
import pandas as pd
import numpy as np
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import median_absolute_error, mean_absolute_percentage_error
from sklearn.metrics import r2_score

In [22]:
df_model = pd.read_csv('data/cleaned_data.csv')
print('Shape:', df_model.shape)

Shape: (397461, 23)


## Section 2 - Train Test Split

In [23]:
latest_month = df_model['CloseYear'] * 100 + df_model['CloseMonth']
test_month = latest_month.max()

df_train = df_model[latest_month != test_month].copy()
df_test = df_model[latest_month == test_month].copy()

X_train = df_train.drop(columns=['ClosePrice'])
y_train = df_train['ClosePrice']
X_test = df_test.drop(columns=['ClosePrice'])
y_test = df_test['ClosePrice']

print(f'Train: {X_train.shape}\nTest: {X_test.shape}')

Train: (384874, 22)
Test: (12587, 22)


### Handling Missing Values for Random Forest

Random Forest in scikit learn does not accept missing values natively. BedBathRatio is imputed with the median from the training set only, consistent with the approach used in the baseline notebook, to avoid data leakage.

In [24]:
median_ratio = X_train['BedBathRatio'].median()
X_train['BedBathRatio'] = X_train['BedBathRatio'].fillna(median_ratio)
X_test['BedBathRatio'] = X_test['BedBathRatio'].fillna(median_ratio)

## Section 3 - Model Training

Random Forest is trained with default hyperparameters as a first pass, before any tuning is applied.

In [25]:
model = RandomForestRegressor(random_state=42, n_jobs=-1)
model.fit(X_train, y_train)

y_pred = model.predict(X_test)
print('Model trained.')

Model trained.


## Section 4 - Model Evaluation

Evaluated using the same metrics as the baseline for direct comparison.

In [26]:
r2 = r2_score(y_test, y_pred)
mape = mean_absolute_percentage_error(y_test, y_pred)
mdape = np.median(np.abs((y_test - y_pred) / y_test))

print(f'R2: {r2:.4f}')
print(f'MAPE: {mape:.4f}')
print(f'MdAPE: {mdape:.4f}')

R2: 0.9036
MAPE: 0.1268
MdAPE: 0.0735


## Section 5 - Summary

Random Forest results on the held out test month (June 2026):

- R2: 0.9036
- MAPE: 0.1268
- MdAPE: 0.0735

This is a substantial improvement over the Linear Regression baseline, where R2 was 0.5468 and MdAPE was 0.2700. The improvement supports the EDA finding that ClosePrice has a non-linear relationship with the available features, which a tree based model captures far more effectively than a linear one.